# PRM Failure Analysis
This notebook:
1. Shows 3 hop-failure examples (questions where retrieval missed a gold paragraph)
2. Shows 3 PRM false-positive prunes (gold paragraphs pruned by the PRM)
3. Plots metric distributions across 500 questions

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
RESULTS_DIR = Path('../results')

## 1. Load raw scores

In [ ]:
df04 = pd.read_csv(RESULTS_DIR / 'raw_scores_t04.csv')
df06 = pd.read_csv(RESULTS_DIR / 'raw_scores_t06.csv')

print('t=0.4 shape:', df04.shape)
print('t=0.6 shape:', df06.shape)
df04.head(3)

## 2. Metric distribution plots

In [ ]:
metrics = ['faithfulness', 'answer_relevancy', 'context_precision',
           'context_recall', 'answer_correctness']

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, metric in zip(axes, metrics):
    if metric in df04.columns:
        ax.hist(df04[metric].dropna(), bins=20, alpha=0.6, label='t=0.4', color='steelblue')
    if metric in df06.columns:
        ax.hist(df06[metric].dropna(), bins=20, alpha=0.6, label='t=0.6', color='coral')
    ax.set_title(metric.replace('_', ' ').title())
    ax.set_xlabel('Score')
    ax.legend(fontsize=8)

plt.tight_layout()
plots_dir = RESULTS_DIR / 'plots'
plots_dir.mkdir(exist_ok=True)
plt.savefig(plots_dir / 'metric_distributions.png', dpi=150)
plt.show()
print('Saved to results/plots/metric_distributions.png')

## 3. Hop-failure examples

A *hop failure* = the gold paragraph was not in the retrieved top-k for that hop.
We identify these by checking whether each gold title appears in `kept` paragraphs.

In [ ]:
# Load per-question pipeline outputs (saved during evaluation)
# Each line in pipeline_outputs.jsonl is a full result dict
pipeline_outputs_path = RESULTS_DIR / 'pipeline_outputs_t04.jsonl'

if pipeline_outputs_path.exists():
    records = []
    with open(pipeline_outputs_path) as f:
        for line in f:
            records.append(json.loads(line))

    # Find hop failures: gold supporting fact not in kept context
    hop_failures = []
    for r in records:
        kept_ids  = {p['id'] for p in r.get('kept', [])}
        gold_ids  = set(r.get('gold_titles', []))
        missed    = gold_ids - kept_ids
        if missed:
            r['missed_gold'] = list(missed)
            hop_failures.append(r)

    print(f'Hop failures: {len(hop_failures)} / {len(records)} ({100*len(hop_failures)/len(records):.1f}%)')

    # Show 3 examples
    for i, ex in enumerate(hop_failures[:3]):
        print(f'\n--- Hop failure {i+1} ---')
        print(f'Question:    {ex["question"]}')
        print(f'Gold answer: {ex["gold_answer"]}')
        print(f'Missed gold: {ex["missed_gold"]}')
        print(f'Model answer:{ex["answer"]}')
else:
    print('pipeline_outputs_t04.jsonl not found — run ragas_eval.py first')
    print('\nExample hop failure (illustrative):')
    print('Q: What is the nationality of the director of the film Inception?')
    print('Gold titles: ["Inception", "Christopher Nolan"]')
    print('Kept titles: ["Inception"]  ← missed "Christopher Nolan" at hop 2')
    print('Model answer: "British" (lucky guess from context)  ← actually got it right')
    print('\nAnalysis: The bridge entity was found but hop-2 sub-question failed to')
    print('retrieve the biography paragraph. This is a sub-question generation failure.')

## 4. PRM false-positive prunes

A *false-positive prune* = the PRM discarded a gold paragraph (PRM score < threshold but paragraph is actually needed).

In [ ]:
if pipeline_outputs_path.exists():
    fp_prunes = []
    for r in records:
        pruned_ids = {p['id'] for p in r.get('pruned', [])}
        gold_ids   = set(r.get('gold_titles', []))
        false_pos  = gold_ids & pruned_ids
        if false_pos:
            r['false_pos_pruned'] = list(false_pos)
            fp_prunes.append(r)

    print(f'False-positive prunes: {len(fp_prunes)} / {len(records)}')

    for i, ex in enumerate(fp_prunes[:3]):
        print(f'\n--- FP prune {i+1} ---')
        print(f'Question:    {ex["question"]}')
        print(f'Pruned gold: {ex["false_pos_pruned"]}')

        # Show the PRM score for the pruned gold paragraph
        for p in ex.get('pruned', []):
            if p['id'] in ex['false_pos_pruned']:
                print(f'PRM score:   {p["prm_score"]:.4f} (threshold was {ex.get("threshold", 0.4)})')
                print(f'Text:        {p["text"][:200]}...')
else:
    print('Run ragas_eval.py first to generate pipeline outputs.')
    print('\nIllustrative example:')
    print('Q: Who is older, Mick Jagger or Keith Richards?')
    print('Gold: "Keith Richards" paragraph (DOB: 18 Dec 1943)')
    print('PRM score: 0.38  (below t=0.4) — pruned!')
    print('Why: The sub-question was "Who is Mick Jagger?" so the PRM gave high')
    print('     score to the Mick Jagger para and low score to Keith Richards.')
    print('Fix: Include the original question (not just sub-question) in PRM scoring.')

## 5. Summary table

In [ ]:
with open(RESULTS_DIR / 'results_t04.json') as f:
    r04 = json.load(f)
with open(RESULTS_DIR / 'results_t06.json') as f:
    r06 = json.load(f)

rows = []
for system, data in [('PRM t=0.4', r04), ('PRM t=0.6', r06)]:
    row = {'System': system}
    for metric, vals in data['metrics'].items():
        row[metric] = vals['ci_str']
    rows.append(row)

summary_df = pd.DataFrame(rows).set_index('System')
summary_df